1) This cell import the required packages for this project to work

In [1]:
from pathlib import Path
from warnings import filterwarnings

# Silence some expected warnings
filterwarnings("ignore")

import pandas as pd
import numpy as np
!pip install rdkit
!pip install scikit-learn
!pip install seaborn
from rdkit import Chem
from rdkit.Chem import MACCSkeys, Draw, rdFingerprintGenerator
from sklearn.model_selection import train_test_split
from sklearn.svm import SVR
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn import metrics
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

%matplotlib inline

   ---------------------------------------- 0.0/24.5 MB ? eta -:--:--
   ------ --------------------------------- 4.2/24.5 MB 25.2 MB/s eta 0:00:01
   ---------------- ----------------------- 10.2/24.5 MB 26.6 MB/s eta 0:00:01
   -------------------------- ------------- 16.0/24.5 MB 26.5 MB/s eta 0:00:01
   ----------------------------------- ---- 21.8/24.5 MB 27.0 MB/s eta 0:00:01
   ---------------------------------------- 24.5/24.5 MB 24.7 MB/s eta 0:00:00



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


2) This cell sets the path correctly so i can refer to it within the notebook

In [2]:
# Set path to current working directory
HERE = Path.cwd()
DATA = HERE / "data"
if not (DATA / "kinase.csv").exists():
    DATA = HERE

3) This cell loads the data from the "kinase.csv" into a dataframe 

In [3]:
# Load data
df = pd.read_csv(DATA / "kinase.csv", index_col=0)
df = df.reset_index(drop=True)

4) This cell prints the shape of the dataframe and executes the .info() function with my df as a parameter. Usefull to get an idea of what type of data the dataframe contains

In [4]:
# Check the dimension and missing value of the data
print("Shape of dataframe : ", df.shape)
df.info()

Shape of dataframe :  (179827, 5)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 179827 entries, 0 to 179826
Data columns (total 5 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   molecule_chembl_id  179827 non-null  object 
 1   standard_value      179827 non-null  float64
 2   standard_units      179827 non-null  object 
 3   target_chembl_id    179827 non-null  object 
 4   smiles              179827 non-null  object 
dtypes: float64(1), object(4)
memory usage: 6.9+ MB


5) This cell prints the head of the dataframe. Usefull to see what the csv and more specifically the data within the csv looks like.

In [5]:
# Look at head
df.head()
# NBVAL_CHECK_OUTPUT

,molecule_chembl_id,standard_value,standard_units,target_chembl_id,smiles
0,CHEMBL13462,4000.0,nM,CHEMBL1862,CC(=O)N[C@@H](Cc1ccc(OP(=O)(O)O)cc1)C(=O)N[C@@...
1,CHEMBL13462,16000.0,nM,CHEMBL1862,CC(=O)N[C@@H](Cc1ccc(OP(=O)(O)O)cc1)C(=O)N[C@@...
2,CHEMBL13462,800.0,nM,CHEMBL267,CC(=O)N[C@@H](Cc1ccc(OP(=O)(O)O)cc1)C(=O)N[C@@...
3,CHEMBL13462,9000.0,nM,CHEMBL267,CC(=O)N[C@@H](Cc1ccc(OP(=O)(O)O)cc1)C(=O)N[C@@...
4,CHEMBL13462,1700.0,nM,CHEMBL267,CC(=O)N[C@@H](Cc1ccc(OP(=O)(O)O)cc1)C(=O)N[C@@...


6) This cell drops the columns i dont need 

In [6]:
# Keep necessary columns
chembl_df = df[["smiles", "standard_value", "standard_units"]].copy()
chembl_df.head()
# NBVAL_CHECK_OUTPUT

,smiles,standard_value,standard_units
0,CC(=O)N[C@@H](Cc1ccc(OP(=O)(O)O)cc1)C(=O)N[C@@...,4000.0,nM
1,CC(=O)N[C@@H](Cc1ccc(OP(=O)(O)O)cc1)C(=O)N[C@@...,16000.0,nM
2,CC(=O)N[C@@H](Cc1ccc(OP(=O)(O)O)cc1)C(=O)N[C@@...,800.0,nM
3,CC(=O)N[C@@H](Cc1ccc(OP(=O)(O)O)cc1)C(=O)N[C@@...,9000.0,nM
4,CC(=O)N[C@@H](Cc1ccc(OP(=O)(O)O)cc1)C(=O)N[C@@...,1700.0,nM


7) This cell is needed to transform the "sandard_value" colunm into pIC50 values. It is much easier to work with pIC50 values. 

In [ ]:
# Transform standard_value in nM to pIC50
chembl_df = chembl_df[chembl_df["standard_units"] == "nM"].copy()
chembl_df = chembl_df[chembl_df["standard_value"] > 0].copy()
chembl_df["pIC50"] = 9 - np.log10(chembl_df["standard_value"])

print("Shape after filtering:", chembl_df.shape)
chembl_df[["standard_value", "standard_units", "pIC50"]].head()
# NBVAL_CHECK_OUTPUT

Shape after filtering: (179154, 4)


,standard_value,standard_units,pIC50
0,4000.0,nM,6.397940
1,16000.0,nM,5.795880
2,800.0,nM,7.096910
3,9000.0,nM,6.045757
4,1700.0,nM,6.769551


8) This cell defines a helper function that converts each SMILES string into a molecular fingerprint. The fingerprint is the numerical representation used as input features for the SVM regressor. It supports different methods, with MACCS used as the default.

In [10]:
def smiles_to_fp(smiles, method="maccs", n_bits=2048):
    """
    Encode a molecule from a SMILES string into a fingerprint.

    Parameters
    ----------
    smiles : str
        The SMILES string defining the molecule.

    method : str
        The type of fingerprint to use. Default is MACCS keys.

    n_bits : int
        The length of the fingerprint.

    Returns
    -------
    array
        The fingerprint array.
    """

    # Convert smiles to RDKit mol object
    mol = Chem.MolFromSmiles(smiles)

    if method == "maccs":
        return np.array(MACCSkeys.GenMACCSKeys(mol))
    if method == "morgan2":
        fpg = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=n_bits)
        return np.array(fpg.GetCountFingerprint(mol))
    if method == "morgan3":
        fpg = rdFingerprintGenerator.GetMorganGenerator(radius=3, fpSize=n_bits)
        return np.array(fpg.GetCountFingerprint(mol))
    else:
        print(f"Warning: Wrong method specified: {method}." " Default will be used instead.")
        return np.array(MACCSkeys.GenMACCSKeys(mol))

9) This cell applies the fingerprint function to every molecule and stores the result in a new fingerprints_df column. It also prints the new shape and previews a few rows to verify feature creation.

In [11]:
chembl_df["fingerprints_df"] = chembl_df["smiles"].apply(smiles_to_fp)

# Look at head
print("Shape of dataframe:", chembl_df.shape)
chembl_df.head(3)
# NBVAL_CHECK_OUTPUT

Shape of dataframe: (179154, 5)


,smiles,standard_value,standard_units,pIC50,fingerprints_df
0,CC(=O)N[C@@H](Cc1ccc(OP(=O)(O)O)cc1)C(=O)N[C@@...,4000.0,nM,6.39794,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,CC(=O)N[C@@H](Cc1ccc(OP(=O)(O)O)cc1)C(=O)N[C@@...,16000.0,nM,5.79588,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2,CC(=O)N[C@@H](Cc1ccc(OP(=O)(O)O)cc1)C(=O)N[C@@...,800.0,nM,7.09691,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


10) This cell splits the fingerprint features and pIC50 targets into training and test sets. A fixed random_state is used so your split is reproducible.

In [12]:
# Build feature matrix and target vector for scikit-learn
X = np.vstack(chembl_df["fingerprints_df"].values).astype(float)
y = chembl_df["pIC50"].values

# Split the data into training and test set
x_train, x_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# Print the shape of training and testing data
print("Shape of training data:", x_train.shape)
print("Shape of test data:", x_test.shape)
# NBVAL_CHECK_OUTPUT

Shape of training data: (125407, 167)
Shape of test data: (53747, 167)


11) This cell defines a reusable SVM regression model using an SVR estimator inside a preprocessing pipeline. Standardization is applied before fitting so all fingerprint dimensions are on a comparable scale.

In [13]:
def svm_model(kernel="rbf", c=10.0, epsilon=0.1, gamma="scale"):
    """Create an SVR model wrapped in a scaling pipeline."""

    return make_pipeline(
        StandardScaler(),
        SVR(kernel=kernel, C=c, epsilon=epsilon, gamma=gamma),
    )

12) This cell sets the SVM hyperparameters used in the following experiments, including candidate kernels and regularization settings.

In [14]:
# SVM parameters
svm_kernels = ["linear", "rbf", "poly"]
svm_c = 10.0
svm_epsilon = 0.1
svm_gamma = "scale"

13) This cell trains one SVM model per kernel choice and compares their regression performance on the test split. It reports MSE and MAE, then visualizes the MAE comparison to help pick the most suitable kernel.

In [ ]:
# Compare different SVM kernels
kernel_results = []
for kernel in svm_kernels:
    model = svm_model(kernel=kernel, c=svm_c, epsilon=svm_epsilon, gamma=svm_gamma)
    model.fit(x_train, y_train)
    pred = model.predict(x_test)
    mse = metrics.mean_squared_error(y_test, pred)
    mae = metrics.mean_absolute_error(y_test, pred)
    kernel_results.append({"kernel": kernel, "mse": mse, "mae": mae})

results_df = pd.DataFrame(kernel_results)
display(results_df)

# Plot kernel comparison
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=results_df, x="kernel", y="mae", ax=ax)
ax.set_title("SVM kernel comparison (lower MAE is better)")
ax.set_ylabel("MAE")
plt.show()

14) This cell trains the final SVM model using the best-performing kernel from the comparison step and saves the fitted model to disk. The saved model can be reloaded later for inference without retraining.

In [ ]:
# Train final SVM model (using best kernel from comparison)
best_kernel = results_df.sort_values("mae").iloc[0]["kernel"]
model = svm_model(kernel=best_kernel, c=svm_c, epsilon=svm_epsilon, gamma=svm_gamma)
model.fit(x_train, y_train)

# Save the trained model
filepath = DATA / "best_svm_model.joblib"
joblib.dump(model, filepath)
print(f"Saved model to: {filepath}")

15) This cell evaluates the trained SVM model on the test set and prints key regression metrics (MSE, MAE, and R2). These values summarize predictive performance after training.

In [ ]:
# Evaluate the model
print("Evaluate the model on the test data")
y_test_pred = model.predict(x_test)
mse = metrics.mean_squared_error(y_test, y_test_pred)
mae = metrics.mean_absolute_error(y_test, y_test_pred)
r2 = metrics.r2_score(y_test, y_test_pred)
print(f"MSE: {mse:.4f}")
print(f"MAE: {mae:.4f}")
print(f"R2:  {r2:.4f}")

16) This cell predicts pIC50 values for the held-out test molecules using the trained SVM model. The first few predictions are printed for a quick sanity check.

In [ ]:
# Predict pIC50 values on x_test data
y_pred = model.predict(x_test)

# Print first 5 predicted values
for value in y_pred[0:5]:
    print(f"{value:.2f}")

17) This cell plots predicted versus true pIC50 values on a scatter plot. The diagonal line indicates ideal agreement between prediction and experiment.

In [ ]:
# Scatter plot
limits = (0, 15)
fig, ax = plt.subplots()
ax.scatter(y_pred, y_test, marker=".")
lin = np.linspace(*limits, 100)
ax.plot(lin, lin)
ax.set_aspect("equal", adjustable="box")
ax.set_xlabel("Predicted values")
ax.set_ylabel("True values")
ax.set_title("Scatter plot: pIC50 values")
ax.set_xlim(limits)
ax.set_ylim(limits)
plt.show()

18) This cell loads an external set of unlabeled compounds from a CSV file. These molecules are used as new candidates for model inference.

In [ ]:
# Load external/unlabeled data set
external_data = pd.read_csv(DATA / "test.csv", index_col=0)
external_data = external_data.reset_index(drop=True)
external_data.head()
# NBVAL_CHECK_OUTPUT

19) This cell converts external SMILES strings into MACCS fingerprints and stores them in a new column. It aligns external inputs with the model's feature format.

In [ ]:
# Convert SMILES strings to MACCS fingerprints
external_data["fingerprints_df"] = external_data["SMILES"].apply(smiles_to_fp)

# Look at head
print("Shape of dataframe : ", external_data.shape)
external_data.head(3)
# NBVAL_CHECK_OUTPUT

20) This cell runs pIC50 predictions on the external compounds and joins them back to the original external dataframe. The resulting table is previewed for inspection.

In [ ]:
# Prediction on external/unlabeled data
external_X = np.vstack(external_data["fingerprints_df"].values).astype(float)
predictions = model.predict(external_X)

predicted_pIC50 = pd.DataFrame(predictions, columns=["predicted_pIC50"])
predicted_pIC50_df = external_data.join(predicted_pIC50)

predicted_pIC50_df.head(5)

21) This cell saves the external prediction results to a CSV file in the data directory. This makes the predictions reusable outside the notebook.

In [ ]:
# Save the predicted values in a csv file in the data folder
predicted_pIC50_df.to_csv(DATA / "predicted_pIC50_df.csv")

22) This cell reloads the saved prediction file, ranks compounds by predicted pIC50, and keeps the top three candidates.

In [ ]:
# Select top 3 drugs
predicted_pIC50_df = pd.read_csv(DATA / "predicted_pIC50_df.csv", index_col=0)
top3_drug = predicted_pIC50_df.nlargest(3, "predicted_pIC50")
top3_drug

23) This cell renders molecular structures for the top-ranked candidates and annotates each with its predicted pIC50. It provides a visual summary of the final selected compounds.

In [ ]:
# Draw the drug molecules
highest_pIC50 = predicted_pIC50_df["SMILES"][top3_drug.index]

mols_EGFR = [Chem.MolFromSmiles(smile) for smile in highest_pIC50]
pIC50_EGFR = top3_drug["predicted_pIC50"].tolist()
pIC50_values = [(f"pIC50 value: {value:.2f}") for value in pIC50_EGFR]

Draw.MolsToGridImage(mols_EGFR, molsPerRow=3, subImgSize=(450, 300), legends=pIC50_values)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NicolaKoch/DSF-NicolaKoch/blob/main/Midterm%20Project/kinase.ipynb)
